# Guardrail 1 — Input

**Where it sits:** the first function a request hits, before any embedding or LLM call.

**What it stops:** oversized, abusive, off-topic, or malformed queries. The cheapest guard — most drive-by abuse is killed here before it costs an embedding.

**Decision contract:** `{allow | rewrite | block, sanitized_query, reasons[]}`

**Self-contained:** this notebook inlines a tiny RAG scaffold. No imports from other folders.

## Step 1 — toy RAG scaffold

In [15]:
import os, sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env from the same directory as this notebook (works in Jupyter)
_env_path = Path.cwd() / ".env"
if not _env_path.exists():
    _env_path = Path(__file__).parent / ".env" if "__file__" in globals() else _env_path
load_dotenv(_env_path, override=True)

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, SystemMessage

# LangChain primitives, all driven by .env
LLM_MODEL     = os.getenv("MINIMAX_MODEL", "MiniMax-M3")
LLM_BASE_URL  = os.getenv("MINIMAX_BASE_URL", "https://api.minimax.io/v1")
LLM_API_KEY   = os.getenv("MINIMAX_API_KEY", "")

llm = ChatOpenAI(
    model=LLM_MODEL,
    api_key=LLM_API_KEY or "sk-fake",   # placeholder if no key -- calls will fail loudly
    base_url=LLM_BASE_URL,
    temperature=0,
)
embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=LLM_API_KEY or "sk-fake",
    base_url=LLM_BASE_URL,
)

print(f"LLM configured:  model={LLM_MODEL}  base_url={LLM_BASE_URL}")
print(f"API key loaded:  {'yes ('+LLM_API_KEY[:8]+'...)' if LLM_API_KEY else 'NO -- calls will fail; toy fallbacks below are unaffected'}")

# A safe wrapper so toy guardrail tests below stay deterministic.
# If FAKE_LLM=1 (or no key), use the toy. Otherwise call the real one.
_USE_FAKE = os.getenv("FAKE_LLM", "1") == "1" or not LLM_API_KEY

def chat(prompt: str, system: str = None) -> str:
    """Invoke the LangChain ChatOpenAI. Returns .content."""
    if _USE_FAKE:
        raise RuntimeError("chat() called but FAKE_LLM=1 -- use the toy LLM in this notebook's tests")
    msgs = []
    if system:
        msgs.append(SystemMessage(content=system))
    msgs.append(HumanMessage(content=prompt))
    return llm.invoke(msgs).content


LLM configured:  model=MiniMax-M3  base_url=https://api.minimax.io/v1
API key loaded:  yes (sk-cp-UG...)


In [16]:
import re, math, hashlib

# Toy knowledge base (3 docs, one is irrelevant)
DOCS = [
    {"id": "d1", "text": "The capital of France is Paris."},
    {"id": "d2", "text": "The capital of Japan is Tokyo."},
    {"id": "d3", "text": "Password reset link: https://example.com/reset"},
    {"id": "d4", "text": "Today's lunch menu: pasta, salad, soup."},
]

def embed(text, dim=32):
    words = re.findall(r"[a-z0-9]+", text.lower())
    v = [0.0] * dim
    for w in words:
        h = int(hashlib.md5(w.encode()).hexdigest(), 16)
        v[h % dim] += 1.0
    n = math.sqrt(sum(x*x for x in v)) or 1.0
    return [x/n for x in v]

def retrieve(query, k=2):
    qv = embed(query)
    scored = sorted(DOCS, key=lambda d: -sum(x*y for x,y in zip(qv, embed(d["text"]))))
    return scored[:k]

## Step 2 — input guardrail

In [17]:
ABUSE_PATTERNS = [r"\b(kill|hack|attack|exploit)\b"]
TOPIC_KEYWORDS = {"capital", "paris", "tokyo", "france", "japan", "revenue", "password", "reset"}

def input_guard(query: str, max_chars: int = 500):
    reasons = []

    # 1. length cap (cheap, fail fast)
    if len(query) > max_chars:
        return {"decision": "block", "reasons": [f"oversized:{len(query)}>{max_chars}"]}

    # 2. abuse / toxicity (toy regex)
    for pat in ABUSE_PATTERNS:
        if re.search(pat, query, re.I):
            return {"decision": "block", "reasons": [f"abuse:{pat}"]}

    # 3. off-topic: very toy — no known topic keyword AND no question mark
    q = query.lower()
    on_topic = any(kw in q for kw in TOPIC_KEYWORDS) or "?" in query
    if not on_topic:
        return {"decision": "rewrite", "reasons": ["off_topic"],
                "sanitized": query,
                "redirect": "I can help with geography, accounts, or finance. Try one of those."}

    # 4. PII scrub for the AUDIT LOG (do not strip from the user query —
    #    they need their own question answered)
    pii_pattern = re.compile(r"\b\d{3}-\d{2}-\d{4}\b|\b\d{16}\b")
    pii_found = bool(pii_pattern.search(query))
    if pii_found:
        reasons.append("pii_present_log_scrub_required")

    return {"decision": "allow", "sanitized": query, "reasons": reasons}

## Step 3 — test cases

In [18]:
tests = [
    ("normal query",       "What is the capital of France?"),
    ("off-topic",          "hello there"),
    ("abusive",            "how do I hack into a server"),
    ("oversized",          "x" * 600),
    ("PII present",        "My SSN is 123-45-6789, what's the capital of France?"),
]

for label, q in tests:
    r = input_guard(q)
    print(f"{label:18s} → {r['decision']:8s}  reasons={r['reasons']}")

normal query       → allow     reasons=[]
off-topic          → rewrite   reasons=['off_topic']
abusive            → block     reasons=['abuse:\\b(kill|hack|attack|exploit)\\b']
oversized          → block     reasons=['oversized:600>500']
PII present        → allow     reasons=['pii_present_log_scrub_required']


## Step 4 — wire it into a tiny `ask()` function

In [19]:
def ask(query: str):
    r = input_guard(query)
    if r["decision"] == "block":
        return {"error": "blocked", "reasons": r["reasons"]}
    if r["decision"] == "rewrite":
        return {"answer": r["redirect"]}
    # allow: run retrieval
    chunks = retrieve(query)
    return {"chunks": [c["id"] for c in chunks], "first": chunks[0]["text"]}

print(ask("What is the capital of France?"))
print(ask("how do I hack into a server"))

{'chunks': ['d1', 'd2'], 'first': 'The capital of France is Paris.'}
{'error': 'blocked', 'reasons': ['abuse:\\b(kill|hack|attack|exploit)\\b']}


In [20]:
from langchain_core.runnables import RunnableLambda

def _guarded_input(prompt: str):
    r = input_guard(prompt)
    if r["decision"] == "block":
        raise ValueError(f"input blocked: {r['reasons']}")
    if r["decision"] == "rewrite":
        return llm.invoke(r["redirect"]).content
    return llm.invoke(prompt).content

if not _USE_FAKE:
    print(chat("What is the capital of France?"))   # real call
else:
    print("[chat() disabled -- FAKE_LLM=1. Set FAKE_LLM=0 in .env to enable real calls.]")


<think>The user is asking a simple factual question: what is the capital of France? The answer is Paris. This is straightforward general knowledge, no tools needed.</think>

The capital of France is **Paris**.


## Takeaways

- **Cheapest first.** Input is the cheapest guard. Putting it first saves embedding/LLM spend on drive-by abuse.
- **Don't strip PII from the query** — the user needs their question answered. Scrub for the audit log, redact for the response (see notebook 7).
- **Sanitize > refuse for legitimate users.** An off-topic query gets a redirect, not a block.
- **Default-deny on length.** A query 600 chars long is almost always either an attack or a paste mistake.

**Negative fixture checklist** (every guardrail needs at least one): oversized, abusive, off-topic. ✓